# Preprocess QC — stage-by-stage retention

Tracks every sample through the pipeline and cross-validates between adjacent stages so silent data loss can't hide.

Stages:
1. **ENA `read_count`** — what the manifest says got deposited
2. **Trimmomatic input** — `Input Read Pairs:` / `Input Reads:` from the .err log
3. **Trimmomatic output** — `Both Surviving:` (PE) or `Surviving:` (SE)
4. **Clumpify input** — `Reads In:` (PE: this counts each end → divide by 2 to compare to pairs)
5. **Clumpify output** — `Reads Out:` post-dedup
6. **QC counted** — `n_r1 + n_r2` from zcat | wc -l on the on-disk fq.gz

Cross-checks (`pct_*` columns):
- `pct_ena_to_trim_in` — ENA→Trim input. ~100% means download wasn't truncated.
- `pct_clump_link` — Trim output → Clumpify input. ~100% means Trimmomatic→Clumpify handoff is intact.
- `pct_qc_link` — Clumpify output → on-disk-counted. ~100% means the file wasn't truncated.

Anything outside [95, 105]% on those three is flagged.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path('../output')
df = pd.read_csv(OUT/'stage_aggregate.tsv', sep='\t')
df['ecotype'] = df['ecotype'].astype(str)
print(f'{len(df)} samples (main={(df.panel=="main").sum()}, loo={(df.panel=="loo").sum()})')
df.head(3)

## 1. Cross-check pass rates

If preprocess is silently broken anywhere, one of these will show <100%.

In [ ]:
checks = pd.DataFrame({
    'check': ['ENA → Trim-In (no truncated DL)',
              'Trim-Out → Clump-In (handoff)',
              'Clump-Out → on-disk QC count (file integrity)'],
    'col':   ['pct_ena_to_trim_in', 'pct_clump_link', 'pct_qc_link'],
    'flag':  ['flag_ena_link', 'flag_clump_link', 'flag_qc_link']
})
rows = []
for _, r in checks.iterrows():
    s = df[r.col].dropna()
    rows.append({
        'check': r.check,
        'n_with_data': len(s),
        'within_95-105': (s.between(95, 105)).sum(),
        'flagged': df[r.flag].fillna(False).sum(),
        'median': s.median(),
        'min': s.min(),
        'max': s.max(),
    })
pd.DataFrame(rows).round(2)

## 2. Trimmomatic survival distribution

How aggressively did trim cull each library? Tail of distribution = candidates for retrim with looser params.

In [ ]:
df['pct_trim_surv'] = df.trim_out / df.trim_in * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, panel in zip(axes, ['main','loo']):
    sub = df[df.panel == panel]
    ax.hist(sub.pct_trim_surv, bins=30, color='#1f4e79', edgecolor='k', alpha=0.85)
    ax.axvline(50, color='r', ls=':', lw=1, label='alert <50%')
    ax.axvline(sub.pct_trim_surv.median(), color='k', ls='--', lw=1, label=f'median {sub.pct_trim_surv.median():.1f}%')
    ax.set_xlabel('Trimmomatic survival %')
    ax.set_title(f'{panel} (n={len(sub)})')
    ax.legend()
plt.tight_layout(); plt.show()

print('Tail of trim survival (lowest 10):')
print(df.nsmallest(10, 'pct_trim_surv')[['panel','ecotype','layout','trim_in','trim_out','pct_trim_surv','dup_pct','cov_est']].to_string(index=False))

## 3. Clumpify dedup distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, panel in zip(axes, ['main','loo']):
    sub = df[df.panel == panel]
    ax.hist(sub.dup_pct, bins=30, color='#d8b365', edgecolor='k', alpha=0.85)
    ax.axvline(50, color='r', ls=':', lw=1, label='alert >50%')
    ax.axvline(sub.dup_pct.median(), color='k', ls='--', lw=1, label=f'median {sub.dup_pct.median():.2f}%')
    ax.set_xlabel('Clumpify duplicate %')
    ax.set_title(f'{panel} (n={len(sub)})')
    ax.legend()
plt.tight_layout(); plt.show()

print('Top 10 by dup rate:')
print(df.nlargest(10, 'dup_pct')[['panel','ecotype','layout','clump_in','clump_out','dup_count','dup_pct','cov_est']].to_string(index=False))

## 4. Stage-by-stage retention per ecotype (stacked bar, sample-by-sample)

Each row of the stacked bar shows: % of ENA reads dropped at trim, % dropped at dedup, % surviving (= what's on disk). Sorted by surviving fraction. Far-right (high survival) is healthy; far-left (low) is over-trimmed.

In [ ]:
for panel in ['main','loo']:
    sub = df[df.panel == panel].copy()
    sub['pct_trim_drop'] = (sub.trim_in - sub.trim_out) / sub.trim_in * 100
    sub['pct_dedup_drop'] = (sub.clump_in_pairs - sub.clump_out_pairs) / sub.trim_in * 100
    sub['pct_surviving']  = sub.clump_out_pairs / sub.trim_in * 100
    sub = sub.sort_values('pct_surviving').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(max(8, len(sub)*0.06), 4.5))
    x = np.arange(len(sub))
    ax.bar(x, sub.pct_surviving,    color='#2ca02c', label='surviving')
    ax.bar(x, sub.pct_dedup_drop,   bottom=sub.pct_surviving, color='#d8b365', label='lost @ dedup')
    ax.bar(x, sub.pct_trim_drop,    bottom=sub.pct_surviving + sub.pct_dedup_drop, color='#d62728', label='lost @ trim')
    # Highlight flagged
    flagged_idx = sub[sub.flag_low_trim | sub.flag_low_cov].index
    for i in flagged_idx:
        ax.axvline(i, color='blue', alpha=0.25, lw=8, zorder=0)
    ax.set_xticks([0, len(sub)//2, len(sub)-1])
    ax.set_xticklabels([sub.ecotype.iloc[i] for i in [0, len(sub)//2, len(sub)-1]])
    ax.set_ylabel('% of Trimmomatic input')
    ax.set_title(f'{panel}: stage-by-stage retention (sorted; blue verticals = flagged)')
    ax.legend(loc='lower right')
    ax.set_ylim(0, 105)
    plt.tight_layout(); plt.show()

## 5. Coverage vs trim survival (where do losses translate to coverage hits?)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for panel, color in [('main', '#1f4e79'), ('loo', '#d8b365')]:
    sub = df[df.panel == panel]
    ax.scatter(sub.pct_trim_surv, sub.cov_est, s=18, alpha=0.7, label=f'{panel} (n={len(sub)})', color=color)
ax.axhline(5, color='r', ls=':', lw=1, label='5x cov floor')
ax.axvline(50, color='r', ls=':', lw=1, label='50% trim floor')
# Annotate flagged samples
flagged = df[df.flag_low_trim | df.flag_low_cov]
for _, r in flagged.iterrows():
    ax.annotate(r.ecotype, (r.pct_trim_surv, r.cov_est), fontsize=9,
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Trimmomatic survival %')
ax.set_ylabel('post-preprocess coverage (×)')
ax.set_title('Trim survival vs final coverage (lower-left = problematic)')
ax.legend()
plt.tight_layout(); plt.show()

## 6. Anomaly summary

In [ ]:
flag_cols = ['flag_ena_link', 'flag_low_trim', 'flag_high_dup',
             'flag_clump_link', 'flag_qc_link', 'flag_low_cov']
summary = df[flag_cols].fillna(False).sum().to_frame('count')
summary['description'] = ['ENA reads ≠ Trim input (truncated DL)',
                          'Trim survival < 50% (over-trim or bad data)',
                          'Dedup > 50% (PCR-heavy library)',
                          'Trim → Clumpify mismatch',
                          'Clumpify → on-disk count mismatch (truncated file)',
                          'Final coverage < 5×']
print(summary)
print()
any_flag = df[flag_cols].fillna(False).any(axis=1)
if any_flag.sum() == 0:
    print('NO ANOMALIES — everything passes cross-checks.')
else:
    print(f'\n{any_flag.sum()} samples with one or more flag set:')
    cols = ['panel','ecotype','layout','read_count','trim_in','trim_out',
            'clump_out_pairs','qc_reads_pairs','cov_est',
            'pct_trim_surv','dup_pct'] + flag_cols
    print(df[any_flag][cols].to_string(index=False))

## 7. Per-sample drill-down (interactive)

In [ ]:
def stage_table(eco):
    r = df[df.ecotype == str(eco)].iloc[0]
    return pd.DataFrame({
        'stage': ['ENA reads', 'Trim input', 'Trim output', 'Clump input (pairs)',
                  'Clump output (pairs)', 'QC reads (pairs)'],
        'count': [r.read_count, r.trim_in, r.trim_out, r.clump_in_pairs,
                  r.clump_out_pairs, r.qc_reads_pairs]
    })

for eco in [9977, 9985, 7521, 7287]:
    print(f'\n=== {eco} ===')
    print(stage_table(eco).to_string(index=False))